# TimbreTrack — a simple example

Extract psychoacoustic timbre features from a `.wav` file with
`comsar.TimbreTrack`, then visualise them: the waveform is drawn in grey with the
feature tracks overlaid in colour, together with a **play button** and a cursor
that follows the audio.

> Requires `bader-comsar` (see the [main README](../README.md)) plus
> `soundfile`, and a browser that can play WAV audio.

In [ ]:
from comsar import TimbreTrack

# used by the visualisation further down
import io, json, base64
from html import escape
import numpy as np
import soundfile as sf
from IPython.display import HTML

In [ ]:
# Initialise the timbre track
tt = TimbreTrack()

In [ ]:
# Extract psychoacoustic features from the recording. The path is relative, so
# the .wav file has to sit next to this notebook. `extract` returns a
# TrackResult; `.features` is a pandas DataFrame with one row per analysis frame.
WAV = "CHI-109_Luguhu_Hulusheng.wav"
result = tt.extract(WAV)
features = result.features
features.head()

## Visualisation

The waveform is drawn in light grey; each extracted feature is normalised to
`[0, 1]` and overlaid in its own colour. Press **Play** to listen — a vertical
cursor follows the playback position, and clicking the plot seeks to that point.
**Click a legend entry** to hide or show that feature; hidden features are shown
in grey in the legend.

The whole player is a self-contained HTML/JavaScript widget (audio embedded as a
data URI), so it also works after the notebook is exported to HTML.

In [ ]:
# --- HTML/JS template for the interactive player ----------------------------
# Runs inside a sandboxed <iframe srcdoc> so the JavaScript also works in
# JupyterLab (which does not execute <script> tags in plain HTML output).
# Click a legend entry to hide/show that feature (hidden = grey text).
_PLAYER_DOC = r"""<!doctype html><html><head><meta charset="utf-8"><style>
  :root{--surface:#fcfcfb;--ink:#0b0b0b;--muted:#52514e;--wave:#d6d6d6;}
  body{margin:0;font:13px/1.4 -apple-system,Segoe UI,Roboto,sans-serif;
       color:var(--ink);background:var(--surface);}
  #wrap{position:relative;}
  canvas{display:block;width:100%;}
  #cursor{position:absolute;top:0;width:2px;background:#0b0b0b;opacity:.55;
          pointer-events:none;left:0;}
  #bar{display:flex;align-items:center;gap:12px;padding:8px 4px;}
  button{font:600 13px sans-serif;border:1px solid #bbb;border-radius:8px;
         background:#fff;padding:6px 14px;cursor:pointer;}
  button:hover{background:#f0f0ef;}
  #t{color:var(--muted);font-variant-numeric:tabular-nums;}
  #legend{display:flex;flex-wrap:wrap;gap:10px 16px;padding:2px 4px 8px;}
  .lg{display:flex;align-items:center;gap:6px;color:var(--muted);
      cursor:pointer;user-select:none;}
  .lg.off{color:#b3b2ae;}
  .lg.off .sw{background:#cfcecb !important;}
  .sw{width:14px;height:3px;border-radius:2px;}
</style></head><body>
  <div id="bar">
    <button id="play">&#9654;&nbsp;Play</button>
    <span id="t">0.00 / 0.00 s</span>
    <span style="color:#8a8a86">— click the plot to seek, click a legend entry to hide/show a feature</span>
  </div>
  <div id="wrap">
    <canvas id="cv"></canvas>
    <div id="cursor"></div>
  </div>
  <div id="legend"></div>
  <audio id="au" src="__AUDIO__" preload="auto"></audio>
<script>
const D = __PAYLOAD__;
const W = D.width, WH = D.waveH, FH = D.featH, H = WH + FH;
for(const s of D.series) s.on = true;
const cv = document.getElementById('cv'), ctx = cv.getContext('2d');
const dpr = window.devicePixelRatio || 1;
cv.width = W*dpr; cv.height = H*dpr; cv.style.height = H+'px'; ctx.scale(dpr,dpr);
function draw(){
  ctx.clearRect(0,0,W,H);
  const midY = WH/2, amp = WH/2*0.92;
  ctx.strokeStyle = getComputedStyle(document.documentElement).getPropertyValue('--wave');
  ctx.lineWidth = 1; ctx.beginPath();
  for(let i=0;i<W;i++){
    ctx.moveTo(i+0.5, midY - D.wmax[i]*amp);
    ctx.lineTo(i+0.5, midY - D.wmin[i]*amp);
  }
  ctx.stroke();
  ctx.strokeStyle = '#ececea'; ctx.lineWidth=1;
  ctx.beginPath(); ctx.moveTo(0,WH+0.5); ctx.lineTo(W,WH+0.5); ctx.stroke();
  const pad = 12, y0 = WH+pad, y1 = H-pad;
  for(const s of D.series){
    if(!s.on) continue;
    const n = s.v.length;
    ctx.strokeStyle = s.color; ctx.lineWidth = 2; ctx.lineJoin='round';
    ctx.beginPath();
    for(let j=0;j<n;j++){
      const x = n>1 ? j/(n-1)*(W-2)+1 : W/2;
      const y = y1 - s.v[j]*(y1-y0);
      j? ctx.lineTo(x,y) : ctx.moveTo(x,y);
    }
    ctx.stroke();
  }
}
draw();
const lg = document.getElementById('legend');
for(const s of D.series){
  const d=document.createElement('span'); d.className='lg';
  d.title='Click to hide/show this feature';
  d.innerHTML='<span class="sw" style="background:'+s.color+'"></span>'+s.name;
  d.onclick=()=>{ s.on=!s.on; d.classList.toggle('off', !s.on); draw(); };
  lg.appendChild(d);
}
const au=document.getElementById('au'), cur=document.getElementById('cursor'),
      btn=document.getElementById('play'), tlab=document.getElementById('t');
cur.style.height = H+'px';
const dur = ()=> (isFinite(au.duration)&&au.duration>0)? au.duration : D.duration;
const fmt = x => x.toFixed(2);
function place(){
  const frac = Math.min(au.currentTime/dur(),1);
  cur.style.left = (frac*100)+'%';
  tlab.textContent = fmt(au.currentTime)+' / '+fmt(dur())+' s';
}
place();
let raf=null;
function loop(){ place(); raf=requestAnimationFrame(loop); }
btn.onclick=()=>{ if(au.paused){au.play();} else {au.pause();} };
au.onplay =()=>{ btn.innerHTML='&#10073;&#10073;&nbsp;Pause'; cancelAnimationFrame(raf); loop(); };
au.onpause=()=>{ btn.innerHTML='&#9654;&nbsp;Play'; cancelAnimationFrame(raf); place(); };
au.onended=()=>{ cancelAnimationFrame(raf); au.currentTime=0; place(); };
cv.style.cursor='pointer';
cv.onclick=(e)=>{ const r=cv.getBoundingClientRect();
  au.currentTime = Math.max(0,Math.min(1,(e.clientX-r.left)/r.width))*dur(); place(); };
</script></body></html>"""


def timbre_player(wav_path, features, width=1000, wave_h=150, feat_h=210):
    """Interactive player: grey waveform + coloured feature curves, a play
    button and a vertical cursor synced to audio playback. Legend entries are
    clickable to hide/show individual features (hidden = grey text). Returns an
    IPython HTML object that renders inline in Jupyter."""
    # audio: waveform min/max envelope, one column per output pixel
    data, sr = sf.read(wav_path, always_2d=True)
    mono = data.mean(axis=1)
    duration = len(mono) / sr
    edges = np.linspace(0, len(mono), width + 1).astype(int)
    wmin = np.zeros(width); wmax = np.zeros(width)
    for i in range(width):
        seg = mono[edges[i]:edges[i + 1]]
        if seg.size:
            wmin[i] = seg.min(); wmax[i] = seg.max()
    peak = max(np.abs(wmin).max(), np.abs(wmax).max(), 1e-9)
    wmin /= peak; wmax /= peak

    # features -> [0, 1] (robust min/max), one coloured curve each
    palette = ["#2a78d6", "#1baf7a", "#eda100", "#008300",
               "#4a3aa7", "#e34948", "#e87ba4"]
    series = []
    for k, name in enumerate(features.columns):
        v = features[name].to_numpy().astype(float)
        lo, hi = np.nanpercentile(v, 2), np.nanpercentile(v, 98)
        if not (np.isfinite(lo) and np.isfinite(hi)) or hi <= lo:
            lo, hi = np.nanmin(v), np.nanmax(v)
            if hi <= lo:
                hi = lo + 1.0
        vn = np.nan_to_num(np.clip((v - lo) / (hi - lo), 0, 1))
        series.append({"name": str(name), "color": palette[k % len(palette)],
                       "v": [round(float(x), 4) for x in vn]})

    # embed a small mono 22.05 kHz WAV as a data URI so it is self-contained
    target_sr = 22050
    if sr != target_sr:
        n_new = int(len(mono) * target_sr / sr)
        m = np.interp(np.linspace(0, len(mono), n_new, endpoint=False),
                      np.arange(len(mono)), mono)
    else:
        m = mono
    m = (m / max(np.abs(m).max(), 1e-9) * 0.98).astype("float32")
    buf = io.BytesIO()
    sf.write(buf, m, target_sr, format="WAV", subtype="PCM_16")
    audio_uri = "data:audio/wav;base64," + base64.b64encode(buf.getvalue()).decode()

    payload = json.dumps({
        "width": width, "waveH": wave_h, "featH": feat_h, "duration": duration,
        "wmin": [round(float(x), 3) for x in wmin],
        "wmax": [round(float(x), 3) for x in wmax],
        "series": series,
    })
    doc = _PLAYER_DOC.replace("__PAYLOAD__", payload).replace("__AUDIO__", audio_uri)
    iframe = ('<iframe style="width:100%;max-width:{w}px;height:{h}px;border:0;'
              'overflow:hidden" sandbox="allow-scripts allow-same-origin" '
              'srcdoc="{doc}"></iframe>').format(
                  w=width + 24, h=wave_h + feat_h + 96, doc=escape(doc, quote=True))
    # wrapped in a <div> so IPython.display.HTML does not emit its
    # "Consider using IPython.display.IFrame instead" warning
    return HTML("<div>" + iframe + "</div>")

In [ ]:
timbre_player(WAV, features)